# DQN on CartPole-v1

Deep Q-Networks bridge tabular Q-Learning and continuous state spaces by approximating `Q(s, a)` with a neural network. Two tricks make it stable:

1. **Experience replay** — break correlation between consecutive samples by shuffling a buffer.
2. **Target network** — a lagged copy of the Q-network used only for computing targets, preventing the "chasing your own tail" problem.

CartPole is a perfect first test: 4 continuous features, 2 actions, solvable in a few hundred episodes.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

from src.agents.dqn import DQNAgent
from src.utils.seeding import set_seed

set_seed(42)
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

agent = DQNAgent(
    obs_dim=obs_dim, n_actions=n_actions,
    hidden_dims=(64, 64), gamma=0.99, lr=1e-3,
    batch_size=32, buffer_size=5000,
    target_update_freq=50,
    epsilon_start=1.0, epsilon_end=0.05, epsilon_decay_steps=2000,
)
print(f"obs_dim={obs_dim}, n_actions={n_actions}")

In [ ]:
rewards, losses = [], []

for ep in range(30):
    state, _ = env.reset()
    ep_reward, ep_loss = 0.0, 0.0
    done = False
    while not done:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        loss = agent.update(state, action, reward, next_state, done)
        ep_reward += reward
        ep_loss += loss
        state = next_state
    rewards.append(ep_reward)
    losses.append(ep_loss)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(rewards)
ax1.set_title('Episode reward (DQN, 30 eps)')
ax1.set_xlabel('Episode')
ax2.plot(losses)
ax2.set_title('Cumulative loss per episode')
ax2.set_xlabel('Episode')
plt.tight_layout()
plt.show()
print(f'Final epsilon: {agent.get_epsilon():.3f}')

30 episodes is just the warmup — the replay buffer is filling up. Run the full 200-episode training with `python -m src.training.trainer --config-name dqn_cartpole` to see the reward curve climb toward 500.